# Experten-Modell HDF - Heat Dissipation Failure

**Manuelle multivariate logistische Regression** (Matrixschreibweise, NumPy + Pandas) fuer den AI4I-2020-Datensatz.

| Modell | Ausfallursache | Physik (Definition im Datensatz) |
|---|---|---|
| **HDF** | Heat Dissipation Failure | (Process_T - Air_T) < 8,6 K UND Drehzahl < 1380 rpm |

Vorgehen:
1. Zwei Features physikalisch begruenden (ggf. neu konstruieren).
2. Manuelle logistische Regression in Matrixform mit Gradient Descent.
3. **Modifizierte Kostenfunktion** mit Klassen-Gewichten, da Ausfaelle stark unterrepraesentiert sind.
4. Trennlinie plotten.
5. Confusion Matrix + F1-Score und Interpretation.


In [ ]:
# Imports - bewusst minimal gehalten.
# sklearn wird NUR zum Vergleichen der manuellen Loesung benutzt, nicht zum Trainieren.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression          # nur zum Vergleich
from sklearn.metrics import confusion_matrix, f1_score       # Standard-Metriken

np.random.seed(42)


In [ ]:
# Daten einlesen
df = pd.read_csv("ai4i2020.csv")
df.head()


In [ ]:
# Wie stark ist HDF unbalanciert? -> begruendet die gewichtete Loss-Funktion.
n_pos = int(df['HDF'].sum())
print(f"HDF: {n_pos:4d} Ausfaelle von {len(df)}  ({n_pos/len(df)*100:5.2f} %)")


## 1. Manuelle logistische Regression in Matrixschreibweise

**Modell** (mit Bias-Spalte 1 in der Designmatrix $X$):

$$
\hat p = \sigma(X\,\theta), \qquad \sigma(z) = \frac{1}{1+e^{-z}}
$$

**Standard-Log-Loss** (ohne Gewichtung):

$$
J(\theta) = -\frac{1}{n}\sum_{i=1}^{n} \Big[ y_i \log \hat p_i + (1-y_i) \log(1-\hat p_i) \Big]
$$

### 2. Modifizierte (gewichtete) Kostenfunktion

Da Ausfaelle nur ca. 0,3 - 1 % der Daten ausmachen, wuerde das Standardmodell fast immer "kein Ausfall" voraussagen und trotzdem eine hohe Accuracy erreichen. Wir gewichten jedes Beispiel mit $w_i$, das von seiner Klasse abhaengt:

$$
w_i = \begin{cases} \dfrac{n}{2\,n_{+}} & \text{falls } y_i = 1 \\[6pt]
                     \dfrac{n}{2\,n_{-}} & \text{falls } y_i = 0 \end{cases}
$$

Damit sind die Gesamtgewichte beider Klassen gleich (entspricht `class_weight='balanced'` in sklearn).
Die gewichtete Loss wird zu

$$
J_w(\theta) = -\frac{1}{\sum_i w_i}\sum_{i=1}^{n} w_i\Big[ y_i \log \hat p_i + (1-y_i) \log(1-\hat p_i) \Big]
$$

und der Gradient (in Matrixform) zu

$$
\nabla_\theta J_w = \frac{1}{\sum_i w_i}\; X^{\top}\big( w \odot (\hat p - y)\big)
$$

Update-Regel (Gradient Descent):  $\theta \leftarrow \theta - \eta \nabla_\theta J_w$.


In [ ]:
# ============================================================
# Bausteine fuer die manuelle logistische Regression
# ============================================================

def sigmoid(z):
    # Numerisch stabile Sigmoid-Funktion (vektorisiert).
    z = np.clip(z, -50, 50)        # vermeidet overflow in exp()
    # Floating-Point-Overflow aus (RuntimeWarning: overflow encountered in exp).
    return 1.0 / (1.0 + np.exp(-z))


def standardize(X):
    # Standardisiert Spalten auf Mittelwert 0 / Standardabweichung 1.
    # Wird manuell gemacht, damit wir die Skalen-Parameter (mu, sigma)
    # fuer die Rueck-Transformation der Trennlinie behalten.
    mu = X.mean(axis=0) # axis=0 -> ueber die Zeilen aggregieren -> ein Mittelwert pro Spalte
    sigma = X.std(axis=0, ddof=0)
    # ddof steht fuer "Delta Degrees of Freedom" und steuert, durch welchen Nenner bei der Standardabweichung geteilt wird.
    # Populations-Standardabweichung   Wenn man die Daten als die gesamte Grundgesamtheit betrachtet
    sigma[sigma == 0] = 1.0 # Sicherheitsnetz gegen Division durch Null.
    return (X - mu) / sigma, mu, sigma


def class_weights(y):
    # 'balanced' Gewichte: jedes Beispiel bekommt n / (2 * n_seine_Klasse).
    n = len(y)
    n_pos = max(int(np.sum(y == 1)), 1) # Anzahl Positive zaehlen, mindestens 1 (Schutz gegen n_pos = 0 -> Division durch Null)
    n_neg = max(int(np.sum(y == 0)), 1)
    w_pos = n / (2.0 * n_pos) # Die zwei Klassen-Gewichte berechnen
    w_neg = n / (2.0 * n_neg)
    # Vektor der Sample-Gewichte (ein Eintrag pro Beobachtung)
    w = np.where(y == 1, w_pos, w_neg)
    # Vektor zusammenbauen: an Stellen wo y=1 steht, kommt w_pos rein, sonst w_neg. Ergibt einen Vektor der Laenge n
    return w


def weighted_log_loss(X_b, y, theta, w):
    # Gewichtete negative Log-Likelihood (Matrixform).
    p = sigmoid(X_b @ theta) # X_b -> Feature-Matix + Bias
    eps = 1e-12
    loss = -np.sum(w * (y*np.log(p+eps) + (1-y)*np.log(1-p+eps))) / np.sum(w)
    return loss


def weighted_gradient(X_b, y, theta, w):
    # Gradient der gewichteten Log-Loss in Matrixform: X^T (w * (p - y))
    p = sigmoid(X_b @ theta)
    return (X_b.T @ (w * (p - y))) / np.sum(w)


def train_logistic(X_std, y, lr=0.1, epochs=5000):
    # Trainings-Schleife: Gradient Descent auf der gewichteten Log-Loss.
    # Erwartet bereits standardisierte Features (ohne Bias-Spalte).
    # Gibt theta = [theta_0, theta_1, theta_2] zurueck.
    n, d = X_std.shape
    X_b = np.c_[np.ones(n), X_std]            # Bias-Spalte anhaengen
    theta = np.zeros(d + 1)                   # Startwert: alle Gewichte 0
    w = class_weights(y)                      # einmalig berechnen
    history = []
    for epoch in range(epochs):
        grad = weighted_gradient(X_b, y, theta, w)
        theta = theta - lr * grad             # GD-Update
        if epoch % 100 == 0:
            history.append(weighted_log_loss(X_b, y, theta, w))
    return theta, history


In [ ]:
# ============================================================
# Plot- und Evaluations-Helfer
# ============================================================

def plot_decision_boundary(X_orig, y, theta, mu, sigma, feat_names, title):
    # Zeichnet Datenpunkte und die im Originalraum zurueck-transformierte Trennlinie.
    # Trennlinie im standardisierten Raum: theta0 + theta1*x1_s + theta2*x2_s = 0
    # => x2_s = -(theta0 + theta1*x1_s)/theta2.
    # Rueck-Transformation: x_orig = x_s * sigma + mu.
    fig, ax = plt.subplots(figsize=(8, 6))

    # Datenpunkte
    ax.scatter(X_orig[y == 0, 0], X_orig[y == 0, 1],
               c='lightgray', s=15, alpha=0.5, label='Kein Ausfall (0)')
    ax.scatter(X_orig[y == 1, 0], X_orig[y == 1, 1],
               c='red', s=35, edgecolor='k', alpha=0.9, label='Ausfall (1)')

    # Trennlinie im standardisierten Raum berechnen ...
    x1_s = np.linspace((X_orig[:, 0].min() - mu[0]) / sigma[0],
                       (X_orig[:, 0].max() - mu[0]) / sigma[0], 200)
    # Schutz vor Division durch (fast) Null
    if abs(theta[2]) < 1e-9:
        ax.set_title(title + "  (theta_2 ~ 0 - keine Linie zeichenbar)")
    else:
        x2_s = -(theta[0] + theta[1] * x1_s) / theta[2]
        # ... und in Originalraum zurueck
        x1_o = x1_s * sigma[0] + mu[0]
        x2_o = x2_s * sigma[1] + mu[1]
        ax.plot(x1_o, x2_o, 'b-', lw=2, label='Entscheidungsgrenze (P=0,5)')

    ax.set_xlim(X_orig[:, 0].min(), X_orig[:, 0].max())
    ax.set_ylim(X_orig[:, 1].min(), X_orig[:, 1].max())
    ax.set_xlabel(feat_names[0])
    ax.set_ylabel(feat_names[1])
    ax.set_title(title)
    ax.legend(loc='best')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate(X_std, y, theta, label):
    # Berechnet Confusion Matrix + F1-Score auf den (standardisierten) Trainingsdaten.
    # Klassifikationsschwelle: 0,5.
    X_b = np.c_[np.ones(len(y)), X_std]
    p = sigmoid(X_b @ theta)
    y_hat = (p >= 0.5).astype(int)

    cm = confusion_matrix(y, y_hat)
    f1 = f1_score(y, y_hat, zero_division=0)

    tn, fp = cm[0, 0], cm[0, 1]
    fn, tp = cm[1, 0], cm[1, 1]
    print(f"--- {label} ---")
    print(f"  Confusion Matrix:")
    print(f"     TN = {tn:5d}    FP = {fp:5d}")
    print(f"     FN = {fn:5d}    TP = {tp:5d}")
    print(f"  F1-Score : {f1:.4f}")
    print(f"  Recall   : {tp/(tp+fn) if (tp+fn)>0 else 0:.4f}   (gefundene Ausfaelle)")
    print(f"  Precision: {tp/(tp+fp) if (tp+fp)>0 else 0:.4f}   (Alarme, die echte Ausfaelle waren)")
    return cm, f1


## HDF - Heat Dissipation Failure

**Physikalische Begruendung der Features**
- *temp_diff = Process T - Air T*: ist der Temperatur-Gradient, ueber den Waerme abgefuehrt werden kann. Kleine Differenz = schlechte Kuehlung. Im Datensatz: HDF ausgeloest bei diff < 8,6 K.
- *Rotational speed [rpm]*: niedrige Drehzahl bedeutet wenig Luftkonvektion am Werkzeug. HDF ausgeloest bei rpm < 1380.

Wir erwarten **beide Koeffizienten negativ**: kleinere temp_diff bzw. niedrigere rpm -> hoehere Ausfallwahrscheinlichkeit.


In [ ]:
# Feature-Engineering: Temperaturdifferenz
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']

features_hdf = ['temp_diff', 'Rotational speed [rpm]']
X_hdf = df[features_hdf].values.astype(float)
y_hdf = df['HDF'].values

X_hdf_s, mu_hdf, sig_hdf = standardize(X_hdf)
theta_hdf, hist_hdf = train_logistic(X_hdf_s, y_hdf, lr=0.1, epochs=5000)

print("Gelernte Koeffizienten [theta_0, theta_1, theta_2] =", theta_hdf)
print("Erwartung: theta_1 (temp_diff) < 0 und theta_2 (rpm) < 0")


In [ ]:
plot_decision_boundary(X_hdf, y_hdf, theta_hdf, mu_hdf, sig_hdf,
                       ['temp_diff = Process - Air [K]', 'Rotational speed [rpm]'],
                       'HDF - Heat Dissipation Failure')
cm_hdf, f1_hdf = evaluate(X_hdf_s, y_hdf, theta_hdf, 'HDF (manuell)')


## Vergleich mit sklearn (zur Validierung der manuellen Implementierung)

Wir trainieren dasselbe Modell zusaetzlich mit `sklearn.LogisticRegression(class_weight='balanced')` und vergleichen den F1-Score. Wenn die manuelle Implementierung korrekt ist, sollten die Werte sehr nahe beieinander liegen.


In [ ]:
# Vergleich: manueller F1 vs. sklearn-F1 fuer HDF
sk = LogisticRegression(class_weight='balanced', max_iter=5000)
sk.fit(X_hdf_s, y_hdf)
f1_sk = f1_score(y_hdf, sk.predict(X_hdf_s), zero_division=0)

print(f"{'Modell':<6}{'F1 manuell':>14}{'F1 sklearn':>14}{'Differenz':>14}")
print('-' * 50)
print(f"{'HDF':<6}{f1_hdf:>14.4f}{f1_sk:>14.4f}{abs(f1_hdf-f1_sk):>14.4f}")


## Interpretation der Ergebnisse

**Was zeigen Confusion Matrix und F1-Score?**

- Die gewichtete Log-Loss zwingt das Modell, die wenigen Ausfaelle ernst zu nehmen, statt sie zugunsten der Mehrheitsklasse "wegzumitteln". Effekt: das manuelle Modell erreicht hohen **Recall** - praktisch jeder echte HDF-Ausfall wird als Alarm erkannt.
- Der Preis dafuer sind **viele False Positives**. Das Modell loest auch dort Alarm aus, wo kein Ausfall stattfindet. Deshalb ist der F1-Wert (Harmonisches Mittel aus Precision und Recall) deutlich kleiner als 1.
- In der Praxis ist dieses Verhalten *gewollt*: ein verpasster Ausfall (False Negative) ist in einer Fabrik teurer als ein falscher Alarm.

**HDF-spezifisch**:

Beide Features sind physikalisch relevant - kleine Temperatur-Differenz **und** niedrige Drehzahl. Die Trennlinie laeuft entsprechend diagonal und das Modell trennt sauber. Die Vorzeichen der Koeffizienten ($\theta_1 < 0$, $\theta_2 < 0$) bestaetigen die physikalische Erwartung.

**Vergleich mit sklearn**:

Der F1-Wert liegt in derselben Groessenordnung, sklearn liegt aber leicht hoeher. Das hat zwei Ursachen:

1. **L2-Regularisierung**: sklearn nutzt per Default eine L2-Strafe (`C=1`), die die Gewichte etwas kleiner haelt. Dadurch wird die Sigmoid weniger steil, der 0,5-Threshold liegt physisch an einer anderen Stelle, und es entstehen weniger False Positives - hoehere Precision, etwas niedrigerer Recall, in Summe oft hoeherer F1.
2. **Optimierer**: sklearn benutzt L-BFGS (Quasi-Newton), wir nutzen einfaches Gradient Descent. L-BFGS konvergiert schneller und genauer zum Optimum der (regularisierten) Loss.

Wichtig ist: das **Vorzeichen und die Groessenordnung der Koeffizienten** stimmen ueberein (bis auf Skalierung), das Modell trifft also dieselbe physikalische Aussage. Das bestaetigt die korrekte Matrix-Implementierung.
